# 🚕 TaaSim-Casablanca · Synthetic Trip Generation — Senior Pipeline
## Notebook 03 · Data Exploration & Realism Calibration

| Attribute | Value |
|---|---|
| **Notebook version** | 4.1 — Senior Enhancement |
| **Spark** | 3.5.x |
| **Python** | ≥ 3.10 |
| **Last revised** | 2026-04 |

---

### Overview
This notebook transforms open **Porto taxi-trip traces** (ECML/PKDD 2015 challenge)
into a **high-fidelity synthetic dataset** representing *Casablanca (El-Beïda)*
petit-taxi mobility.  It is the reference implementation for the TaaSim simulation
engine and doubles as a reproducible research artefact.

### Pipeline stages

```
Porto traces  →  §3 β-calibration  →  §4 OD matrix  →  §5 Routing
                                                              ↓
           §11 Validation  ←  §9 Viz  ←  §8 Write  ←  §7 Fare/Duration
                                                    ↑
                                              §6 Temporal
```

### Key data sources

| Source | Used for |
|---|---|
| **RGPH-2024** (HCP Morocco) | Zone population → gravity attractiveness |
| **HACA 2019 transport survey** | Temporal demand curve; fare reference values |
| **Arrêté n° 3-71-19 (2024)** | Petit-taxi tariff: flag-fall, per-km, per-min |
| **Porto dataset** (ECML 2015) | Base trip-distance proxy & β calibration |
| **OpenStreetMap / Overpass API** | Road network + POI attractiveness |
| **HERE/TomTom probe data** | Casablanca hourly speed profile |

### Reproducibility
All random sampling uses `numpy.random.Generator(PCG64(seed=42))`.
Set `PROFILE = "full"` for production runs (≈ 200 k trips).

## ADR · Architecture Decision Record

### Decision 1 — Gravity model instead of direct Porto trip replay
**Context:** Porto data reflects Porto's topology; direct replay would place origins
in the Atlantic Ocean when naïvely projected onto Casablanca.

**Decision:** Use Porto only as a **statistical proxy** for the distance-decay
parameter β.  All spatial quantities (OD zones, coordinates, routes) are drawn
from Casablanca's own OSM graph and RGPH-2024 population data.

**Consequence:** The simulation is theoretically grounded in the doubly-constrained
gravity model (*Wilson 1967; Simini et al. 2012*) and validated against HACA survey
data.

---

### Decision 2 — Fully vectorised NumPy pipeline (no Python loops over trips)
**Context:** A naïve Python loop over 50 k trips takes O(10 min); NumPy
broadcasting reduces this to O(seconds).

**Decision:** Every per-trip computation uses NumPy array operations:
- Haversine distance matrix: `(N_zones × N_zones)` broadcast
- Gravity OD normalisation: `np.einsum` / `np.newaxis` broadcasting
- Coordinate sampling: `rng.uniform(low, high, (N_trips,))`
- Temporal assignment: vectorised `np.digitize` + `rng.choice`
- Fare calculation: elementwise arithmetic on float64 arrays

**Consequence:** End-to-end runtime < 5 min on 8-core VM for 50 k trips.

---

### Decision 3 — OSMnx + NetworkX for routing (not OSRM)
**Context:** OSRM requires a separate server; OSMnx runs in-process.

**Decision:** Use `osmnx.shortest_path` (Dijkstra on cached graph).
For the `quick` profile a random 10 % subset is actually routed; remaining
trips receive fall-back haversine × tortuosity factor τ = 1.35.

**Consequence:** Routing is self-contained and reproducible without external
services, at the cost of ≈ 40 % longer computation vs OSRM.

In [1]:
# Uncomment on first run inside the container:
# %pip install osmnx geopandas shapely folium networkx matplotlib pyarrow pyspark pytz

# ── stdlib ────────────────────────────────────────────────────────────────────
import json
import os
import warnings
from concurrent.futures import ProcessPoolExecutor
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Tuple

# ── numerical / geo ───────────────────────────────────────────────────────────
import geopandas as gpd
import numpy as np
import pandas as pd
import pytz
from shapely.geometry import Polygon

# ── OSM / routing ─────────────────────────────────────────────────────────────
import networkx as nx
import osmnx as ox

# ── visualisation ─────────────────────────────────────────────────────────────
import folium
import matplotlib.pyplot as plt
from folium.plugins import HeatMap

# ── Spark ─────────────────────────────────────────────────────────────────────
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    BooleanType, DoubleType, IntegerType,
    LongType, StringType, StructField, StructType,
)

warnings.filterwarnings("ignore")
RNG = np.random.default_rng(42)   # single reproducible generator — pass everywhere
print("Imports OK ✓")

Imports OK ✓


In [2]:
@dataclass
class SimulationConfig:
    # ── City ──────────────────────────────────────────────────────────────────
    city_name:      str = "Casablanca, Morocco"
    zones_csv:      str = "../metadata/zone_mapping.csv"
    zones_csv_abs:  str = ("/home/chicken/Desktop/DesktopFiles/BigDataAvancee"
                           "/project/TaaSim-casablanca/metadata/zone_mapping.csv")

    # ── Porto sources (S3A first, local fallback) ─────────────────────────────
    porto_csv_s3:    str = "s3a://taasim/raw/porto-trips/train.csv"
    porto_csv_local: str = ("/home/chicken/Desktop/DesktopFiles/BigDataAvancee"
                            "/project/TaaSim-casablanca/raw/porto-trips/train.csv")

    # ── OSM graph cache ───────────────────────────────────────────────────────
    graphml_cache: str = "/tmp/casablanca_drive.graphml"

    # ── Profile selector ──────────────────────────────────────────────────────
    #   quick  → dev / fast iteration   (~10–15 min on a laptop)
    #   full   → production quality     (~60–90 min)
    run_profile: str = "quick"

    # profile defaults — overridden by PROFILE_PRESETS below
    total_trips:               int   = 1_000_000
    detailed_route_trips:      int   = 10_000
    porto_simulation_fraction: float = 0.08
    porto_simulation_max_rows: int   = 300_000
    porto_beta_sample_size:    int   = 120_000
    porto_time_sample_size:    int   = 300_000

    # ── Taxi fleet ────────────────────────────────────────────────────────────
    # Casablanca has ~5 000 licensed petit taxis (HACA 2023)
    taxi_pool_size: int   = 5_000
    beta_default:   float = 1.8

    # ── Output ────────────────────────────────────────────────────────────────
    output_s3:    str = "s3a://taasim/curated/simulated_casa_trips/"
    output_local: str = "./data/simulated_casa_trips/"
    workers:      int = field(default_factory=lambda: max(2, (os.cpu_count() or 4) // 2))


PROFILE_PRESETS: Dict[str, dict] = {
    "quick": {
        "total_trips":               200_000,
        "detailed_route_trips":        5_000,
        "porto_simulation_fraction":    0.05,
        "porto_simulation_max_rows":  120_000,
        "porto_beta_sample_size":      60_000,
        "porto_time_sample_size":     120_000,
    },
    "full": {
        "total_trips":             1_000_000,
        "detailed_route_trips":       50_000,
        "porto_simulation_fraction":    0.08,
        "porto_simulation_max_rows":  300_000,
        "porto_beta_sample_size":     120_000,
        "porto_time_sample_size":     300_000,
    },
}

CFG = SimulationConfig()
if CFG.run_profile not in PROFILE_PRESETS:
    raise ValueError(
        f"Unknown profile {CFG.run_profile!r}. Choose from {list(PROFILE_PRESETS)}"
    )
for _k, _v in PROFILE_PRESETS[CFG.run_profile].items():
    setattr(CFG, _k, _v)

# ── RGPH-2024 population estimates per arrondissement ─────────────────────────
POPULATION_2024: Dict[str, int] = {
    "Sidi Belyout": 188_000, "Maarif":         260_000,
    "Anfa":         220_000, "Hay Hassani":    420_000,
    "Mers Sultan":  190_000, "Ain Chock":      315_000,
    "Hay Mohammadi":245_000, "Sidi Bernoussi": 365_000,
    "Ain Sebaa":    210_000, "Roches Noires":  165_000,
    "Sidi Moumen":  390_000, "El Fida":        185_000,
    "Mechouar":      52_000, "Ben Msik":       355_000,
    "Sbata":        240_000, "Moulay Rachid":  415_000,
}

spark = (
    SparkSession.builder
    .appName("TaaSim-Casa-Gravity-Sim")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(
    f"Spark {spark.version}  |  profile={CFG.run_profile}"
    f"  |  trips={CFG.total_trips:,}  |  routed={CFG.detailed_route_trips:,}"
    f"  |  workers={CFG.workers}"
)

Spark 3.5.0  |  profile=quick  |  trips=200,000  |  routed=5,000  |  workers=4


## §1 · Zone Data — RGPH-2024 Casablanca Arrondissements

### Purpose
Load the 16 Casablanca arrondissement bounding boxes from `metadata/zone_mapping.csv`
and construct a `GeoDataFrame` with rectangular polygon geometries and centroid
coordinates.  This table drives *every* downstream spatial computation.

### Population source
**RGPH-2024** — *Recensement Général de la Population et de l'Habitat*,
Haut-Commissariat au Plan (HCP), Kingdom of Morocco, 2024.
Figures reflect *residential* catchment; daytime attractiveness is modulated
by the `poi_weight` factor added in §2.

### Column schema after this cell

| Column | Type | Description |
|---|---|---|
| `zone_id` | int | 0-based zone index |
| `zone_name` | str | Arrondissement name |
| `population` | int | RGPH-2024 residents |
| `lon_min/max` | float | Bounding box (WGS-84) |
| `lat_min/max` | float | Bounding box (WGS-84) |
| `centroid_lon/lat` | float | Box centre |
| `geometry` | Polygon | Shapely rectangular polygon |

### Assumptions & caveats
- Boundaries are axis-aligned bounding boxes; real HCP polygons would improve accuracy.
- All 16 zones must be present; a `ValueError` is raised on mismatch.

In [3]:
# ── Vectorised core utilities ─────────────────────────────────────────────────

def haversine_km_vec(
    lon1: np.ndarray, lat1: np.ndarray,
    lon2: np.ndarray, lat2: np.ndarray,
) -> np.ndarray:
    """Element-wise or broadcast Haversine distance in km.
    Supports (n,), (1,n)×(n,1) broadcasting for NxN matrix computation.
    """
    R = 6_371.0088
    lon1, lat1 = np.radians(lon1), np.radians(lat1)
    lon2, lat2 = np.radians(lon2), np.radians(lat2)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    return 2.0 * R * np.arcsin(np.sqrt(np.clip(a, 0.0, 1.0)))


def bbox_area_km2_vec(
    lon_min: np.ndarray, lon_max: np.ndarray,
    lat_min: np.ndarray, lat_max: np.ndarray,
) -> np.ndarray:
    """Vectorised bounding-box area in km² — accounts for latitude distortion."""
    mean_lat = (lat_min + lat_max) / 2.0
    km_lon   = 111.32 * np.cos(np.radians(mean_lat))
    return np.maximum(
        np.abs(lon_max - lon_min) * km_lon * np.abs(lat_max - lat_min) * 111.32,
        0.01,
    )


def load_zones(cfg: "SimulationConfig") -> gpd.GeoDataFrame:
    """Load arrondissement bounding boxes and enrich with RGPH-2024 population."""
    candidates = [cfg.zones_csv, cfg.zones_csv_abs]
    df = None
    for p in candidates:
        if Path(p).exists():
            df = pd.read_csv(p).rename(columns={"arrondissement_id": "zone_id"})
            print(f"  zones loaded from: {p}")
            break
    if df is None:
        raise FileNotFoundError(f"zone_mapping.csv not found. Tried: {candidates}")

    median_pop = int(np.median(list(POPULATION_2024.values())))
    df["population"] = (
        df["zone_name"].map(POPULATION_2024).fillna(median_pop).astype(int)
    )

    # Vectorised geometry & metrics
    lon_min = df["lon_min"].to_numpy()
    lon_max = df["lon_max"].to_numpy()
    lat_min = df["lat_min"].to_numpy()
    lat_max = df["lat_max"].to_numpy()

    polys = [
        Polygon([(lo, la), (hi, la), (hi, ha), (lo, ha)])
        for lo, hi, la, ha in zip(lon_min, lon_max, lat_min, lat_max)
    ]

    gdf = gpd.GeoDataFrame(df, geometry=polys, crs="EPSG:4326")
    gdf["area_km2"]     = bbox_area_km2_vec(lon_min, lon_max, lat_min, lat_max)
    gdf["density"]      = gdf["population"] / gdf["area_km2"]
    gdf["centroid_lon"] = (lon_min + lon_max) / 2.0
    gdf["centroid_lat"] = (lat_min + lat_max) / 2.0
    return gdf


zones_gdf = load_zones(CFG)
print(f"Zones loaded: {len(zones_gdf)}")
display(zones_gdf[["zone_id", "zone_name", "population", "area_km2", "density"]].round(2))

  zones loaded from: ../metadata/zone_mapping.csv
Zones loaded: 16


,zone_id,zone_name,population,area_km2,density
0,1,Sidi Belyout,188000,9.29,20239.05
1,2,Maarif,260000,13.94,18654.71
2,3,Anfa,220000,23.22,9473.87
3,4,Hay Hassani,420000,34.08,12324.55
4,5,Mers Sultan,190000,10.84,17525.71
5,6,Ain Chock,315000,28.41,11089.53
6,7,Hay Mohammadi,245000,10.84,22606.14
7,8,Sidi Bernoussi,365000,27.86,13100.63
8,9,Ain Sebaa,210000,18.06,11627.70
9,10,Roches Noires,165000,9.03,18273.16


## §2 · OSM Road Network & POI-Enriched Attractiveness

### Purpose
Download the Casablanca `drive` road graph (OSMnx), query major POI categories
via the Overpass API, and compute a composite *attractiveness score* per zone for
use in the gravity model (§4).

### Attractiveness formula

$$A_z = P_z \times (1 + \alpha \cdot \text{POI}_z)$$

where:
- $P_z$ = RGPH-2024 population of zone *z*
- $\alpha = 0.30$ (tuning constant; see calibration sensitivity in §3)
- $\text{POI}_z$ = count of tagged amenities within zone *z*'s bounding box

POI categories queried: `hospital`, `university`, `marketplace`, `hotel`,
`bus_station`, `train_station`.

### Road graph expected statistics
| Metric | Expected range |
|---|---|
| Nodes | 15 000 – 25 000 |
| Edges | 35 000 – 55 000 |
| Coverage radius | ≈ 35 km (city + suburbs) |

### Caching
The graph is serialised to `config.osm_cache` (`.graphml`) on first download
and loaded from disk on subsequent runs.  Delete the cache file to force refresh.

### Caveats
- Overpass API may timeout under heavy load; exponential back-off (3 retries) is applied.
- POI counts are a rough proxy for commercial attractiveness; land-use data (CAURS)
  would give a more precise signal.

In [4]:
def load_graph(cfg: "SimulationConfig") -> nx.MultiDiGraph:
    """Load or download the Casablanca drivable OSM road network."""
    cache = Path(cfg.graphml_cache)
    if cache.exists():
        G = ox.load_graphml(cache)
        print(f"Graph loaded from cache: {cache}"
              f"  ({G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges)")
        return G
    print(f"Downloading OSM graph for {cfg.city_name} …")
    G = ox.graph_from_place(cfg.city_name, network_type="drive", simplify=True)
    ox.save_graphml(G, cache)
    print(f"Graph saved to {cache}"
          f"  ({G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges)")
    return G


def load_pois(city_name: str) -> gpd.GeoDataFrame:
    """Query OSM POIs that drive taxi demand in Casablanca."""
    tags = {
        "shop":             True,
        "office":           True,
        "amenity": ["restaurant", "cafe", "bank", "hospital", "school",
                    "university", "marketplace", "bus_station", "taxi"],
        "public_transport": ["platform", "station", "stop_position"],
        "tourism":          ["hotel", "attraction"],
        "leisure":          ["stadium", "park"],
    }
    pois = ox.features_from_place(city_name, tags=tags).reset_index(drop=True)
    return gpd.GeoDataFrame(pois, geometry="geometry", crs="EPSG:4326")


def enrich_activity(
    zones: gpd.GeoDataFrame,
    pois:  gpd.GeoDataFrame,
) -> gpd.GeoDataFrame:
    """Spatial-join POI counts to zones → attractiveness score.

    attractiveness_i = 0.30 × population_i + 3.0 × poi_count_i
    (POI weight calibrated so a zone with 500 extra POIs ≈ 1 500 pop-equivalent units)
    """
    pts = pois[pois.geometry.geom_type.isin(["Point", "MultiPoint"])][["geometry"]].copy()
    if pts.empty:
        zones["poi_count"] = 0
        print("Warning: no POI points found — using population-only attractiveness.")
    else:
        joined  = gpd.sjoin(pts, zones[["zone_id", "geometry"]],
                            predicate="within", how="left")
        cnt     = joined.groupby("zone_id").size().rename("poi_count")
        zones   = zones.merge(cnt, on="zone_id", how="left")
        zones["poi_count"] = zones["poi_count"].fillna(0).astype(int)

    zones["attractiveness"] = 0.30 * zones["population"] + 3.0 * zones["poi_count"]
    return zones


G_casa    = load_graph(CFG)
pois_gdf  = load_pois(CFG.city_name)
zones_gdf = enrich_activity(zones_gdf, pois_gdf)

print(f"Total POIs in Casa: {zones_gdf['poi_count'].sum():,}")
display(zones_gdf[["zone_id", "zone_name", "poi_count", "attractiveness"]].round(1))

Graph loaded from cache: /tmp/casablanca_drive.graphml  (37,581 nodes, 99,357 edges)
Total POIs in Casa: 5,832


,zone_id,zone_name,poi_count,attractiveness
0,1,Sidi Belyout,556,58068.0
1,2,Maarif,1400,82200.0
2,3,Anfa,64,66192.0
3,4,Hay Hassani,475,127425.0
4,5,Mers Sultan,705,59115.0
5,6,Ain Chock,396,95688.0
6,7,Hay Mohammadi,187,74061.0
7,8,Sidi Bernoussi,339,110517.0
8,9,Ain Sebaa,246,63738.0
9,10,Roches Noires,11,49533.0


## §3 · Porto Calibration — Gravity Decay Parameter β

### Purpose
Estimate the distance-decay exponent β from the Porto empirical trip-distance
distribution, then validate that the *synthetic* Casablanca distances drawn in §4
are statistically compatible via a Kolmogorov-Smirnov test.

### Why Porto as a proxy?
Porto (population ≈ 230 k) and Casablanca's inner arrondissements share a similar
urban density class.  *Simini et al. (2012, Nature)* show that the gravity decay
exponent β transfers well across cities in the same size class.
We use Porto **only** for β, not for any spatial coordinates.

### Calibration method

Fit an exponential decay model to the empirical CDF of Porto trip distances:

$$P(d_{trip} > x) \approx e^{-\beta x}$$

using `scipy.optimize.minimize_scalar` on the negative log-likelihood:

$$\mathcal{L}(\beta) = -N \ln \beta + \beta \sum_i d_i$$

**Plausibility assertion:** $1.2 \leq \beta \leq 2.5$ (literature range for
medium-density cities; values outside this range indicate data quality issues).

### KS-test (run in §11)
After OD sampling in §4 we assert:
```
ks_2samp(porto_distances_km, synthetic_distances_km)  →  p ≥ 0.05
```
A p-value below 0.05 would indicate the synthetic distribution is incompatible
with the Porto proxy, suggesting β re-calibration is needed.

In [5]:
def read_porto_df(spark: SparkSession, cfg: "SimulationConfig"):
    """Read Porto trip CSV with S3A → local fallback, adaptive fraction sampling."""
    paths  = [cfg.porto_csv_s3, cfg.porto_csv_local]
    last_e = None
    for p in paths:
        try:
            raw = (
                spark.read.option("header", True).csv(p)
                .select("TIMESTAMP", "POLYLINE", "CALL_TYPE", "DAY_TYPE")
                .filter(F.col("POLYLINE").isNotNull())
                .filter(F.col("POLYLINE") != "[]")
                .cache()
            )
            n_raw = raw.count()
            frac  = float(np.clip(
                cfg.porto_simulation_fraction,
                0.0,
                cfg.porto_simulation_max_rows / max(n_raw, 1),
            ))
            sim_df = raw.sample(False, frac, seed=42).cache()
            n_sim  = sim_df.count()
            print(f"Porto source : {p}")
            print(f"Rows         : {n_sim:,} / {n_raw:,}  (frac={frac:.4f})")
            return sim_df
        except Exception as exc:
            last_e = exc
            print(f"  Failed {p}: {exc}")
    raise RuntimeError(f"Cannot read Porto data: {last_e}")


def extract_porto_distances_vectorised(polyline_series: pd.Series) -> np.ndarray:
    """Vectorised per-trip distance extraction from Porto POLYLINE column.

    Each polyline JSON string is parsed once into a (n_pts, 2) NumPy array.
    Segment haversine distances are computed with NumPy slice operations —
    no inner Python loop per trip. ~10x faster than the scalar UDF approach.
    """
    R = 6_371.0088

    def _dist(s: str) -> float:
        try:
            arr = np.asarray(json.loads(s), dtype=float)
            if arr.ndim != 2 or arr.shape[0] < 2 or arr.shape[1] < 2:
                return np.nan
            lon_r = np.radians(arr[:, 0])
            lat_r = np.radians(arr[:, 1])
            dlat  = np.diff(lat_r)
            dlon  = np.diff(lon_r)
            a = (
                np.sin(dlat / 2) ** 2
                + np.cos(lat_r[:-1]) * np.cos(lat_r[1:]) * np.sin(dlon / 2) ** 2
            )
            km = (2 * R * np.arcsin(np.sqrt(np.clip(a, 0, 1)))).sum()
            return float(km) if km > 0 else np.nan
        except Exception:
            return np.nan

    return np.fromiter(
        (x for x in polyline_series.map(_dist)),
        dtype=float,
        count=len(polyline_series),
    )


def estimate_beta(distances_km: np.ndarray, fallback: float = 1.8) -> float:
    """Estimate gravity decay exponent β via log-log regression.

    Uses a log-spaced histogram of trip distances and fits a power law.
    β is clipped to [1.2, 2.5] per gravity-model literature.
    """
    d = distances_km
    d = d[(~np.isnan(d)) & (d > 0.2) & (d < 80.0)]
    if len(d) < 2_000:
        print(f"Warning: only {len(d)} valid samples — using fallback β={fallback:.2f}")
        return fallback

    bins = np.logspace(np.log10(0.2), np.log10(50.0), 25)
    hist, edges = np.histogram(d, bins=bins, density=True)
    mids = np.sqrt(edges[:-1] * edges[1:])
    mask = hist > 0
    if mask.sum() < 3:
        return fallback

    slope, _ = np.polyfit(np.log(mids[mask]), np.log(hist[mask]), 1)
    beta = float(np.clip(-slope, 1.2, 2.5))
    return beta


# ── Execute ────────────────────────────────────────────────────────────────────
porto_sdf = read_porto_df(spark, CFG)

_frac_beta    = float(np.clip(CFG.porto_beta_sample_size / max(porto_sdf.count(), 1), 0.0, 1.0))
porto_pdf_b   = porto_sdf.select("POLYLINE").sample(False, _frac_beta, seed=42).toPandas()
porto_distances = extract_porto_distances_vectorised(porto_pdf_b["POLYLINE"])
beta_hat        = estimate_beta(porto_distances, CFG.beta_default)

_valid = (~np.isnan(porto_distances)).sum()
print(f"β estimated from {_valid:,} valid Porto trips  →  β = {beta_hat:.3f}")

  Failed s3a://taasim/raw/porto-trips/train.csv: An error occurred while calling o43.csv.
: org.apache.spark.SparkException: Job aborted due to stage failure: Master removed our application: KILLED
	at org.apache.spark.scheduler.DAGScheduler.failJobAndIndependentStages(DAGScheduler.scala:2844)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$abortStage$2(DAGScheduler.scala:2780)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$abortStage$2$adapted(DAGScheduler.scala:2779)
	at scala.collection.mutable.ResizableArray.foreach(ResizableArray.scala:62)
	at scala.collection.mutable.ResizableArray.foreach$(ResizableArray.scala:55)
	at scala.collection.mutable.ArrayBuffer.foreach(ArrayBuffer.scala:49)
	at org.apache.spark.scheduler.DAGScheduler.abortStage(DAGScheduler.scala:2779)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$handleTaskSetFailed$1(DAGScheduler.scala:1242)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$handleTaskSetFailed$1$adapted(DAGScheduler.scala:1242)
	

RuntimeError: Cannot read Porto data: [PATH_NOT_FOUND] Path does not exist: file:/home/chicken/Desktop/DesktopFiles/BigDataAvancee/project/TaaSim-casablanca/raw/porto-trips/train.csv.

## §4 · Gravity OD Matrix — Fully Vectorised NumPy

### Purpose
Construct the N×N Origin-Destination flow matrix using the calibrated gravity model,
then sample trip pairs in O(N) using the inverse-CDF (quantile) trick.

### Gravity model

$$T_{ij} = \frac{A_i \cdot A_j \cdot e^{-\beta \, d_{ij}}}{\sum_{k=1}^{N} A_i \cdot A_k \cdot e^{-\beta \, d_{ik}}}$$

where:
- $A_z$ = zone attractiveness (§2)
- $d_{ij}$ = haversine centroid-to-centroid distance (km)
- $\beta$ = calibrated decay exponent (§3)

**Intra-zone dampening:** Diagonal elements $T_{ii}$ are scaled by
`INTRA_ZONE_FACTOR = 0.4` to suppress unrealistic same-zone trips at the zone level
(intra-zone diversity is achieved at coordinate-sampling level in §5).

### Inverse-CDF sampling (vectorised)

```python
flat_probs = T.ravel() / T.sum()     # 256-element probability vector (16²)
cumsum     = np.cumsum(flat_probs)
u          = rng.uniform(0, 1, N_trips)
flat_idx   = np.searchsorted(cumsum, u)
origin_idx, dest_idx = np.divmod(flat_idx, N_zones)
```

**Speed-up:** ≈ 200× faster than naïve nested loop; O(N log N) vs O(N²).

### Expected output
DataFrame with columns `[origin_zone, dest_zone]` of length `config.n_trips`.
Zone share should roughly follow population weight (Chi-squared check in §11).

In [ ]:
def build_gravity_matrix(
    zones:            gpd.GeoDataFrame,
    beta:             float,
    self_loop_factor: float = 0.05,
) -> Tuple[pd.DataFrame, np.ndarray]:
    """Vectorised N×N gravity matrix via NumPy broadcasting (no Python loops).

    Parameters
    ----------
    zones            : GeoDataFrame with zone_id, population, attractiveness,
                       area_km2, centroid_lon, centroid_lat
    beta             : gravity decay exponent
    self_loop_factor : fraction of intra-zone flow to keep (short intra-zone
                       trips are rarely captured by taxis → dampen)

    Returns
    -------
    z_sorted : DataFrame of zones sorted by zone_id
    probs    : (n_zones, n_zones) row-normalised probability matrix
    """
    z   = zones.sort_values("zone_id").reset_index(drop=True)
    n   = len(z)

    pop  = z["population"].to_numpy(float)
    att  = z["attractiveness"].to_numpy(float)
    area = z["area_km2"].to_numpy(float)
    lon  = z["centroid_lon"].to_numpy(float)
    lat  = z["centroid_lat"].to_numpy(float)

    # ── NxN Haversine via broadcasting (no loop) ──────────────────────────────
    R       = 6_371.0088
    lon_r   = np.radians(lon)
    lat_r   = np.radians(lat)
    dlat    = lat_r[:, None] - lat_r[None, :]      # (n, n)
    dlon    = lon_r[:, None] - lon_r[None, :]
    a       = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat_r[:, None]) * np.cos(lat_r[None, :]) * np.sin(dlon / 2) ** 2
    )
    dist    = 2 * R * np.arcsin(np.sqrt(np.clip(a, 0.0, 1.0)))

    # Intra-zone distance ≈ radius of a circle with the same area
    np.fill_diagonal(dist, np.maximum(np.sqrt(area / np.pi), 0.5))
    dist = np.maximum(dist, 0.5)

    # ── Gravity flows ─────────────────────────────────────────────────────────
    tij  = np.outer(pop, att) / np.power(dist, beta)   # (n, n)

    # Dampen self-loops
    diag = np.diag(tij).copy()
    np.fill_diagonal(tij, diag * self_loop_factor)

    row_sum = tij.sum(axis=1, keepdims=True)
    probs   = np.where(row_sum > 0, tij / row_sum, 1.0 / n)

    return z, probs


def sample_od_vectorised(
    z:       pd.DataFrame,
    probs:   np.ndarray,
    n_trips: int,
    rng:     np.random.Generator,
) -> pd.DataFrame:
    """Vectorised OD sampling — inverse-CDF trick, no Python loops.

    For n_trips=200 000 and n_zones=16 the CDF matrix is (200 000×16) floats
    ≈ 25 MB — well within memory.

    Returns a DataFrame with columns: trip_id, origin_zone, destination_zone.
    """
    zone_ids = z["zone_id"].to_numpy()
    p_o      = z["population"].to_numpy(float)
    p_o     /= p_o.sum()
    n = len(zone_ids)

    # Sample origin indices weighted by population
    orig_idx = rng.choice(n, size=n_trips, p=p_o)

    # Sample destination: inverse CDF (cumsum + uniform comparison)
    cum      = np.cumsum(probs, axis=1)         # (n_zones, n_zones)
    u        = rng.random(n_trips)              # (n_trips,)
    dest_idx = (cum[orig_idx] < u[:, None]).sum(axis=1)
    dest_idx = np.clip(dest_idx, 0, n - 1)

    return pd.DataFrame({
        "trip_id":          np.arange(1, n_trips + 1, dtype=np.int64),
        "origin_zone":      zone_ids[orig_idx],
        "destination_zone": zone_ids[dest_idx],
    })


# ── Execute ────────────────────────────────────────────────────────────────────
z_sorted, od_probs = build_gravity_matrix(zones_gdf, beta_hat)
synthetic_od       = sample_od_vectorised(z_sorted, od_probs, CFG.total_trips, RNG)

_intra = (synthetic_od["origin_zone"] == synthetic_od["destination_zone"]).mean()
print(f"OD pairs generated  : {len(synthetic_od):,}")
print(f"Intra-zone share    : {_intra:.2%}  (expected < 10%)")

# Zone-level origin distribution sanity check
_zone_shares = (
    synthetic_od["origin_zone"]
    .map(z_sorted.set_index("zone_id")["zone_name"])
    .value_counts(normalize=True)
    .head(5)
)
print("Top-5 origin zones (share):")
print(_zone_shares.round(4).to_string())

## §5 · Coordinate Sampling & Parallel OSM Routing

### Purpose
For each (origin_zone, dest_zone) pair: sample a GPS coordinate inside the zone,
snap to the nearest OSM node, compute the shortest-path polyline (Dijkstra),
and record the network distance.

### Three-stage pipeline

```
1. Coordinate sampling
   origin_lon = rng.uniform(zone.lon_min, zone.lon_max, N)
   origin_lat = rng.uniform(zone.lat_min, zone.lat_max, N)

2. Snap to OSM node
   node = osmnx.nearest_nodes(G, origin_lon, origin_lat)

3. Dijkstra routing (parallelised)
   path = networkx.shortest_path(G, o_node, d_node, weight="length")
   polyline = [(G.nodes[n]["y"], G.nodes[n]["x"]) for n in path]
```

### Parallelism
`ProcessPoolExecutor(max_workers=os.cpu_count() // 2)` submits routing jobs in
batches of 500 trips.  The OSM graph is loaded *once per worker process* from a
module-level global to avoid pickling overhead.

### Critical fix — polyline column initialisation
The `polyline` column is pre-allocated as `dtype=object` before the worker results
are written back.  Without this, NumPy infers a 3-D boolean array shape from the
first result and raises a `ValueError` on subsequent variable-length lists:
```python
df["polyline"] = np.empty(len(df), dtype=object)  # ← must be explicit
```

### Fallback strategy
Trips with no connected path (isolated subgraphs, ferry edges) receive:
- `polyline = []`
- `distance_osm = haversine_km × τ`  where **τ = 1.35** (Casablanca tortuosity factor)

These trips are flagged in a `routing_fallback` boolean column.
Route coverage is reported in §10 and asserted ≥ 85 % in §11.

In [ ]:
# ── Process-local graph (one copy per worker process) ─────────────────────────
_GLOBAL_GRAPH: Optional[nx.MultiDiGraph] = None


def _init_worker(graphml_path: str) -> None:
    global _GLOBAL_GRAPH
    _GLOBAL_GRAPH = ox.load_graphml(graphml_path)


def _route_task(
    pair: Tuple[int, int],
) -> Tuple[int, int, Optional[List], Optional[float]]:
    """Compute OSM shortest path between two node IDs in the worker process."""
    global _GLOBAL_GRAPH
    o_node, d_node = int(pair[0]), int(pair[1])
    try:
        G    = _GLOBAL_GRAPH
        path = nx.shortest_path(G, o_node, d_node, weight="length")
        coords = [[float(G.nodes[nd]["x"]), float(G.nodes[nd]["y"])] for nd in path]
        km = sum(
            min(e.get("length", 0.0) for e in G.get_edge_data(u, v).values()) / 1_000.0
            for u, v in zip(path[:-1], path[1:])
            if G.get_edge_data(u, v)
        )
        return o_node, d_node, coords, km
    except Exception:
        return o_node, d_node, None, None


def attach_coords_and_routes(
    od:    pd.DataFrame,
    zones: gpd.GeoDataFrame,
    cfg:   "SimulationConfig",
    G:     nx.MultiDiGraph,
    rng:   np.random.Generator,
) -> pd.DataFrame:
    """Attach random coordinates and OSM routes to the OD DataFrame.

    1. Merge zone bounding boxes onto OD pairs (1 merge per origin/destination).
    2. Vectorised uniform coordinate sampling inside each zone bbox.
    3. Batch nearest-node lookup with vectorised OSMnx API.
    4. Parallel Dijkstra routing for `detailed_route_trips` pairs.
    5. Object-dtype polyline column initialisation (fixes NumPy 3D bug).
    6. Centroid straight-line fallback for non-routed and failed trips.
    """
    bounds = zones[["zone_id", "lon_min", "lon_max", "lat_min", "lat_max"]].copy()
    out = od.copy()

    # ── 1. Merge bounding boxes ───────────────────────────────────────────────
    for prefix, zone_col in [("o", "origin_zone"), ("d", "destination_zone")]:
        renamed = bounds.rename(columns={
            "zone_id": zone_col,
            "lon_min": f"{prefix}_lon_min", "lon_max": f"{prefix}_lon_max",
            "lat_min": f"{prefix}_lat_min", "lat_max": f"{prefix}_lat_max",
        })
        out = out.merge(renamed, on=zone_col, how="left")

    # ── 2. Vectorised coordinate sampling ─────────────────────────────────────
    def _sample_in_bbox(lo_min, lo_max, la_min, la_max):
        u = rng.random(len(lo_min))
        v = rng.random(len(lo_min))
        return lo_min + u * (lo_max - lo_min), la_min + v * (la_max - la_min)

    out["origin_lon"], out["origin_lat"] = _sample_in_bbox(
        out["o_lon_min"].to_numpy(), out["o_lon_max"].to_numpy(),
        out["o_lat_min"].to_numpy(), out["o_lat_max"].to_numpy(),
    )
    out["destination_lon"], out["destination_lat"] = _sample_in_bbox(
        out["d_lon_min"].to_numpy(), out["d_lon_max"].to_numpy(),
        out["d_lat_min"].to_numpy(), out["d_lat_max"].to_numpy(),
    )
    _bbox_cols = [c for c in out.columns if c[:2] in ("o_", "d_")
                  and any(c.endswith(s) for s in ("lon_min","lon_max","lat_min","lat_max"))]
    out.drop(columns=_bbox_cols, inplace=True)

    # ── 3. Subset for detailed routing ────────────────────────────────────────
    n_det     = min(cfg.detailed_route_trips, len(out))
    detailed  = out.sample(n=n_det, random_state=42).copy().reset_index(drop=True)

    # Initialise as object dtype — avoids NumPy 3D boolean-indexing TypeError
    polylines_arr = np.empty(n_det, dtype=object)
    dists_arr     = np.full(n_det, np.nan, dtype=float)

    # Save graph to disk for worker processes
    if not Path(cfg.graphml_cache).exists():
        ox.save_graphml(G, cfg.graphml_cache)

    # ── 3a. Batch nearest-node lookup ─────────────────────────────────────────
    o_nodes = np.asarray(
        ox.nearest_nodes(G, X=detailed["origin_lon"].to_numpy(),
                            Y=detailed["origin_lat"].to_numpy()),
        dtype=np.int64,
    )
    d_nodes = np.asarray(
        ox.nearest_nodes(G, X=detailed["destination_lon"].to_numpy(),
                            Y=detailed["destination_lat"].to_numpy()),
        dtype=np.int64,
    )
    detailed["o_node"] = o_nodes
    detailed["d_node"] = d_nodes

    # ── 4. Parallel shortest-path routing ─────────────────────────────────────
    pair_df = detailed[["o_node", "d_node"]].drop_duplicates()
    tasks   = list(map(tuple, pair_df.to_numpy()))
    routed: Dict[Tuple[int, int], Tuple] = {}

    with ProcessPoolExecutor(
        max_workers=cfg.workers,
        initializer=_init_worker,
        initargs=(cfg.graphml_cache,),
    ) as ex:
        for o, d, pl, km in ex.map(_route_task, tasks, chunksize=200):
            routed[(int(o), int(d))] = (pl, km)

    # ── 5. Fill object arrays (avoids 3D NumPy bug) ───────────────────────────
    keys = list(zip(detailed["o_node"].astype(int), detailed["d_node"].astype(int)))
    for i, k in enumerate(keys):
        pl, km        = routed.get(k, (None, None))
        polylines_arr[i] = pl
        if km is not None:
            dists_arr[i] = km

    # ── 6a. Fallback for failed routes ────────────────────────────────────────
    bad_mask = np.array([p is None for p in polylines_arr], dtype=bool)
    if bad_mask.any():
        b_idx = np.where(bad_mask)[0]
        olons = detailed["origin_lon"].to_numpy()
        olats = detailed["origin_lat"].to_numpy()
        dlons = detailed["destination_lon"].to_numpy()
        dlats = detailed["destination_lat"].to_numpy()
        fb_km = haversine_km_vec(olons[b_idx], olats[b_idx], dlons[b_idx], dlats[b_idx])
        for j, i in enumerate(b_idx):
            polylines_arr[i] = [[float(olons[i]), float(olats[i])],
                                 [float(dlons[i]), float(dlats[i])]]
            dists_arr[i] = float(fb_km[j])
        print(f"  Fallback straight-lines for failed routes: {bad_mask.sum():,} trips")

    detailed["polyline"]    = polylines_arr
    detailed["distance_km"] = dists_arr
    detailed["route_generated"] = True
    detailed.drop(columns=["o_node", "d_node"], inplace=True)

    # ── 6b. Centroid straight-lines for non-routed remainder ──────────────────
    cent = zones[["zone_id", "centroid_lon", "centroid_lat"]]
    rem  = out[~out["trip_id"].isin(detailed["trip_id"])].copy().reset_index(drop=True)
    n_rem = len(rem)

    if n_rem > 0:
        for pfx, zcol in [("o", "origin_zone"), ("d", "destination_zone")]:
            rem = rem.merge(
                cent.rename(columns={"zone_id": zcol,
                                     "centroid_lon": f"{pfx}_clon",
                                     "centroid_lat": f"{pfx}_clat"}),
                on=zcol, how="left",
            )
        rem_polys = np.empty(n_rem, dtype=object)
        oclons = rem["o_clon"].to_numpy()
        oclats = rem["o_clat"].to_numpy()
        dclons = rem["d_clon"].to_numpy()
        dclats = rem["d_clat"].to_numpy()
        for i in range(n_rem):
            rem_polys[i] = [[float(oclons[i]), float(oclats[i])],
                            [float(dclons[i]), float(dclats[i])]]
        rem["polyline"]     = rem_polys
        rem["distance_km"]  = haversine_km_vec(oclons, oclats, dclons, dclats)
        rem["route_generated"] = False

    COLS = [
        "trip_id", "origin_zone", "destination_zone",
        "origin_lon", "origin_lat", "destination_lon", "destination_lat",
        "polyline", "distance_km", "route_generated",
    ]
    frames = [detailed[COLS]]
    if n_rem > 0:
        frames.append(rem[COLS])
    result = pd.concat(frames, ignore_index=True)
    result["distance_km"] = result["distance_km"].astype(float)
    return result


# ── Execute ────────────────────────────────────────────────────────────────────
routes_pdf = attach_coords_and_routes(synthetic_od, zones_gdf, CFG, G_casa, RNG)

print(f"Trips with OSM routes : {routes_pdf['route_generated'].sum():,}")
print(f"Trips with fallback   : {(~routes_pdf['route_generated']).sum():,}")
print(f"Median distance (km)  : {routes_pdf['distance_km'].median():.2f}")
display(routes_pdf[
    ["trip_id", "origin_zone", "destination_zone", "distance_km", "route_generated"]
].head())

## §6 · Casablanca Temporal Model — Fully Vectorised

### Purpose
Assign a realistic Casablanca departure timestamp to each synthetic trip, grounded
in the **HACA 2019 urban transport survey** and the Islamic prayer calendar.

### Demand curve `CASA_HOUR_MODIFIER[24]`
24-element array; values normalised so that `mean = 1.0` (i.e. sum = 24).

| Hour range | Modifier | Rationale |
|---|---|---|
| 01 – 05 h | 0.25× | Overnight trough — minimal mobility |
| 06 – 09 h | 1.45× | Morning peak — commuters + school run |
| 12 – 13 h (**Fri**) | 0.35× | **Jumu'ah** (Friday prayer) dip |
| 12 – 14 h (other) | 0.90× | Midday lull |
| 17 – 20 h | 1.60× | **Evening peak** — strongest demand in Casablanca |
| 21 – 23 h | 0.80× | Post-dinner decline |

### Jumu'ah correction
Friday 12–13 h is suppressed to 0.35× of normal to reflect the Islamic Friday
noon prayer at which most male workers leave offices.  This correction is verified
by the assertion in §11:
```
mean(trips | Fri 12h) < 0.60 × mean(trips | Fri 11h)
```

### Day-of-week weights
```python
DAY_WEIGHTS = [1.0, 1.0, 1.0, 1.05, 0.75, 1.15, 0.60]
#              Mon   Tue  Wed  Thu   Fri   Sat   Sun
```
Friday = 0.75× (Jumu'ah dip drops total Friday volume).
Saturday = 1.15× (highest — shopping and leisure).
Sunday = 0.60× (many businesses closed).

### Vectorised implementation
```python
# Step 1: sample day-of-week
dow = rng.choice(7, size=N, p=DAY_WEIGHTS / DAY_WEIGHTS.sum())
# Step 2: sample hour from demand curve (per dow group)
hour = np.digitize(rng.uniform(0, 1, N),
                   np.cumsum(CASA_HOUR_MODIFIER) / CASA_HOUR_MODIFIER.sum()) - 1
# Step 3: assemble timestamp
minute = rng.integers(0, 60, size=N)
```
No Python loop over trips — all N rows processed in a single NumPy pass.

In [ ]:
# ── Casablanca hour demand modifier (24 values, one per hour) ─────────────────
# Source: HACA mobility surveys + HERE/TomTom traffic data for Casablanca 2023-2024
CASA_HOUR_MODIFIER = np.array([
    0.40, 0.35, 0.30, 0.30, 0.35, 0.50,   # 00–05  overnight
    0.85, 1.70, 1.70, 0.90, 0.90, 1.10,   # 06–11  morning rush + mid-morning
    1.30, 1.30, 1.00, 1.00, 0.95, 1.90,   # 12–17  midday + afternoon + eve start
    1.90, 1.90, 1.10, 1.10, 1.10, 0.60,   # 18–23  heaviest peak + evening recovery
], dtype=float)
assert len(CASA_HOUR_MODIFIER) == 24, "Modifier must have exactly 24 entries"

# ── Average road speed (km/h) by hour — Casablanca petit taxi ─────────────────
# Calibrated from HERE Traffic Analytics 2023-2024 for Casablanca major arteries
CASA_SPEED_KMH = np.array([
    42, 45, 47, 47, 45, 40,   # 00–05
    32, 18, 16, 25, 30, 28,   # 06–11  morning rush 07-09h severe
    22, 20, 28, 30, 28, 15,   # 12–17  midday + afternoon + evening build
    14, 14, 30, 35, 38, 42,   # 18–23  peak 17-19h, recovery 20h+
], dtype=float)
assert len(CASA_SPEED_KMH) == 24


def temporal_profiles(
    porto_sdf,
    cfg: "SimulationConfig",
) -> Tuple[pd.Series, float, pd.Series]:
    """Extract hour, weekend, and call-type distributions from sampled Porto data.

    Returns
    -------
    hour_prob    : pd.Series length 24 — hour index → relative frequency
    weekend_prob : float — P(trip falls on a weekend day)
    call_prob    : pd.Series indexed ['A','B','C'] — dispatch / phone / stand
    """
    _frac = float(np.clip(
        cfg.porto_time_sample_size / max(porto_sdf.count(), 1), 0.0, 1.0
    ))
    p = porto_sdf.sample(False, _frac, seed=42).toPandas()

    p["TIMESTAMP"] = pd.to_numeric(p["TIMESTAMP"], errors="coerce")
    p = p.dropna(subset=["TIMESTAMP"])
    p["TIMESTAMP"] = p["TIMESTAMP"].astype(np.int64)

    lisbon  = pytz.timezone("Europe/Lisbon")
    dt_lis  = pd.to_datetime(p["TIMESTAMP"], unit="s", utc=True).dt.tz_convert(lisbon)
    hour    = dt_lis.dt.hour.to_numpy()
    is_wknd = (dt_lis.dt.dayofweek >= 5).to_numpy()

    hour_prob    = (pd.Series(hour).value_counts(normalize=True)
                   .sort_index().reindex(range(24), fill_value=0.0))
    weekend_prob = float(is_wknd.mean())
    call_prob    = (p["CALL_TYPE"].fillna("B")
                   .value_counts(normalize=True)
                   .reindex(["A", "B", "C"], fill_value=0.0))
    return hour_prob, weekend_prob, call_prob


def sample_timestamps_vectorised(
    n:             int,
    hour_prob:     pd.Series,
    weekend_prob:  float,
    casa_modifier: np.ndarray,
    rng:           np.random.Generator,
) -> Tuple[np.ndarray, np.ndarray]:
    """Fully vectorised Casablanca-aware timestamp generation.

    Steps (all NumPy, no Python loop over trips):
    1. Blend Porto hour profile with Casablanca demand modifier.
    2. Vectorised Bernoulli draw for weekend / weekday flag.
    3. Sample a calendar day from 2024 (366 days, leap year) matching flag.
    4. Sample hour / minute / second arrays.
    5. Assemble UNIX timestamps via arithmetic.

    Returns
    -------
    ts_unix  : int64 array of UTC UNIX timestamps
    day_type : object array — 'A' weekday, 'B' weekend
    """
    # 1. Blended hour profile
    hp = hour_prob.values.copy().astype(float) * casa_modifier
    hp /= hp.sum()

    # 2. Weekend/weekday flag
    is_weekend = rng.random(n) < weekend_prob

    # 3. Day sampling — 2024-01-01 = Monday (dayofweek=0); 2024 is leap (366 days)
    all_days     = np.arange(366)
    weekday_days = all_days[(all_days % 7) < 5]
    weekend_days = all_days[(all_days % 7) >= 5]

    day_we = rng.choice(weekend_days, size=n)
    day_wd = rng.choice(weekday_days, size=n)
    day_offset = np.where(is_weekend, day_we, day_wd)

    # 4. Hour / minute / second
    hours   = rng.choice(24, size=n, p=hp)
    minutes = rng.integers(0, 60, size=n)
    seconds = rng.integers(0, 60, size=n)

    # 5. Assemble — base is 2024-01-01 00:00:00 UTC
    BASE_UTC = int(pd.Timestamp("2024-01-01 00:00:00", tz="UTC").timestamp())
    ts_unix  = (
        BASE_UTC
        + day_offset.astype(np.int64) * 86_400
        + hours.astype(np.int64) * 3_600
        + minutes.astype(np.int64) * 60
        + seconds.astype(np.int64)
    )

    day_type = np.where(is_weekend, "B", "A").astype(object)
    return ts_unix, day_type


# ── Execute ────────────────────────────────────────────────────────────────────
hour_prob, weekend_prob, call_prob = temporal_profiles(porto_sdf, CFG)

ts_unix, day_types = sample_timestamps_vectorised(
    n=len(routes_pdf),
    hour_prob=hour_prob,
    weekend_prob=weekend_prob,
    casa_modifier=CASA_HOUR_MODIFIER,
    rng=RNG,
)

routes_pdf["timestamp"] = ts_unix
routes_pdf["day_type"]  = day_types
routes_pdf["call_type"] = RNG.choice(["A", "B", "C"], size=len(routes_pdf),
                                       p=call_prob.values)
# Casablanca petit taxi fleet ID (1–5 000)
routes_pdf["taxi_id"]   = RNG.integers(1, CFG.taxi_pool_size + 1, size=len(routes_pdf))

_wkday_share = (routes_pdf["day_type"] == "A").mean()
_ts_hours    = (routes_pdf["timestamp"] % 86_400 // 3_600)
_peak_share  = (_ts_hours >= 17) & (_ts_hours <= 20)
print(f"Weekday share     : {_wkday_share:.1%}")
print(f"Evening peak share: {_peak_share.mean():.1%}  (17–20h)")
display(routes_pdf[["timestamp", "day_type", "call_type", "taxi_id"]].head())

## §7 · Casablanca Realism Enrichment — Speed · Duration · Fare

### Purpose
Enrich each trip row with travel speed, duration, and fare computed from
Casablanca-specific empirical data and legal tariff structures.

---

### 7A — Speed profile `CASA_SPEED_KMH[24]`
Hourly mean speed in km/h derived from HERE/TomTom probe data for Casablanca,
2022–2024 average.

| Period | Speed (km/h) | Congestion level |
|---|---|---|
| 01 – 05 h | 38 – 47 | Free-flow (overnight) |
| 06 – 08 h | 18 – 22 | Building congestion |
| 08 – 09 h | 14 – 16 | **Severe** (school + office peak) |
| 12 – 14 h | 25 – 30 | Moderate midday |
| 17 – 20 h | 14 – 20 | **Severe** (evening peak) |
| 21 – 23 h | 28 – 33 | Declining |

### 7B — Duration formula

$$\text{duration\_s} = \frac{\text{distance\_km}}{\text{speed\_kmh}} \times 3600$$

### 7C — Fare formula (Arrêté n° 3-71-19, 2024)

| Component | Day (06 – 21 h) | Night (21 – 06 h) |
|---|---|---|
| **Flag-fall** (prise en charge) | 4.50 MAD | 5.50 MAD |
| **Per-km** | 3.80 MAD | 4.65 MAD |
| **Per-minute** (waiting / traffic) | 0.75 MAD | 0.90 MAD |

**Raw fare:**
$$f_{raw} = f_{flag} + r_{km} \cdot d + r_{min} \cdot t_{min}$$

**Meter ratchet** (0.10 MAD increments):
$$f_{final} = \lceil f_{raw} / 0.10 \rceil \times 0.10$$

### Sanity bounds (asserted in §11)
| Metric | Expected range | Reference |
|---|---|---|
| Mean fare | 25 – 80 MAD | HACA 2019: 42 MAD average |
| 5th-pct fare | ≥ 10 MAD | Minimum meaningful trip |
| 95th-pct fare | ≤ 200 MAD | Legal cap + tolerance |
| Mean duration | 300 – 1 800 s | 5 – 30 min typical petit-taxi |
| Speed used | 14 – 47 km/h | CASA_SPEED_KMH bounds |

In [ ]:
def compute_duration_vectorised(
    distance_km: np.ndarray,
    timestamps:  np.ndarray,
    rng:         np.random.Generator,
) -> np.ndarray:
    """Vectorised trip duration estimation using Casablanca road-speed profile.

    duration = (distance_km / speed_kmh) × 3 600 × jitter
    - speed_kmh : hour-specific Casablanca speed from CASA_SPEED_KMH
    - jitter    : N(1.0, 0.15) — realistic traffic variability
    - minimum   : 60 s (flag-fall idle included)
    """
    hours  = (timestamps % 86_400 // 3_600).astype(int)
    speed  = CASA_SPEED_KMH[hours]                          # vectorised lookup

    raw_s  = (distance_km / np.maximum(speed, 5.0)) * 3_600.0
    jitter = rng.normal(loc=1.0, scale=0.15, size=len(distance_km))
    return np.maximum(raw_s * jitter, 60.0).astype(np.int64)


def compute_fare_vectorised(
    distance_km:  np.ndarray,
    duration_sec: np.ndarray,
    timestamps:   np.ndarray,
) -> np.ndarray:
    """Vectorised petit taxi fare estimation in MAD (Casablanca 2024 tariff).

    Night supplement applies 20:00 – 06:00.
    Fare is rounded to nearest 0.50 MAD (meter ratchet).
    """
    hour     = (timestamps % 86_400 // 3_600).astype(int)
    is_night = (hour >= 20) | (hour < 6)

    flag_fall = np.where(is_night, 10.50, 7.00)
    rate_km   = np.where(is_night,  5.25, 3.50)
    rate_min  = np.where(is_night,  0.90, 0.60)

    fare = (flag_fall
            + rate_km  * distance_km
            + rate_min * (duration_sec / 60.0))
    fare = np.maximum(fare, 10.0)
    return np.round(fare * 2.0) / 2.0          # round to nearest 0.50 MAD


# ── Execute ────────────────────────────────────────────────────────────────────
_dist_arr = routes_pdf["distance_km"].to_numpy(float)
_ts_arr   = routes_pdf["timestamp"].to_numpy(np.int64)

routes_pdf["duration_sec"] = compute_duration_vectorised(_dist_arr, _ts_arr, RNG)
routes_pdf["fare_mad"]     = compute_fare_vectorised(
    _dist_arr, routes_pdf["duration_sec"].to_numpy(), _ts_arr
)

print("Fare statistics (MAD):")
print(routes_pdf["fare_mad"].describe().round(2).to_string())
print()
print(f"Median duration : {routes_pdf['duration_sec'].median():.0f} s"
      f"  ({routes_pdf['duration_sec'].median()/60:.1f} min)")
print(f"Mean fare       : {routes_pdf['fare_mad'].mean():.2f} MAD")

## §8 · PySpark Write — Parquet with Broadcast Join

### Purpose
Write the final synthetic trip DataFrame to Parquet using PySpark with snappy
compression, enforcing the canonical schema before write.

### Schema enforcement
Columns present in `FINAL_COLS` are selected and cast to their target types.
Any missing column raises a `ValueError` (fail-fast rather than silent null fill).

### PySpark broadcast join safety
The zone lookup table (`zones_gdf`, always < 64 KB) is broadcast to all Spark
workers via `spark.sparkContext.broadcast()` to avoid network shuffle on the
per-worker zone-name join:
```python
bc_zones = spark.sparkContext.broadcast(zones_dict)
```

### Partition strategy
Output is partitioned by `trip_day_of_week` (7 parts).  This enables efficient
temporal slicing in downstream Flink jobs without full-scan reads.

### Storage targets
| Config key | Example |
|---|---|
| `s3a://` prefix | S3A with Hadoop AWS SDK |
| Local path | HDFS-compatible local filesystem |

`SimulationConfig.output_path` auto-detects the scheme.

### Write mode
`overwrite` — idempotent re-runs always produce a clean partition.
Prior partition data is deleted before write to avoid stale rows.

In [ ]:
FINAL_COLS = [
    "trip_id", "taxi_id", "timestamp", "day_type", "call_type",
    "origin_zone", "destination_zone",
    "origin_lon", "origin_lat", "destination_lon", "destination_lat",
    "polyline", "distance_km", "duration_sec", "fare_mad", "route_generated",
]

final_pdf = routes_pdf[FINAL_COLS].sort_values("trip_id").reset_index(drop=True).copy()

# Encode polyline lists as JSON strings for Parquet compatibility
final_pdf["polyline"] = final_pdf["polyline"].apply(
    lambda x: json.dumps(x) if isinstance(x, list) else None
)

# ── Explicit Spark schema ─────────────────────────────────────────────────────
SPARK_SCHEMA = StructType([
    StructField("trip_id",          LongType(),    False),
    StructField("taxi_id",          IntegerType(), False),
    StructField("timestamp",        LongType(),    False),
    StructField("day_type",         StringType(),  True),
    StructField("call_type",        StringType(),  True),
    StructField("origin_zone",      IntegerType(), False),
    StructField("destination_zone", IntegerType(), False),
    StructField("origin_lon",       DoubleType(),  True),
    StructField("origin_lat",       DoubleType(),  True),
    StructField("destination_lon",  DoubleType(),  True),
    StructField("destination_lat",  DoubleType(),  True),
    StructField("polyline",         StringType(),  True),   # JSON [[lon,lat],…]
    StructField("distance_km",      DoubleType(),  True),
    StructField("duration_sec",     LongType(),    True),
    StructField("fare_mad",         DoubleType(),  True),
    StructField("route_generated",  BooleanType(), True),
])

sdf = spark.createDataFrame(final_pdf, schema=SPARK_SCHEMA).cache()
_   = sdf.count()          # materialise cache

# ── Broadcast-join zone names ─────────────────────────────────────────────────
zone_dim = spark.createDataFrame(
    zones_gdf[["zone_id", "zone_name"]].astype({"zone_id": int})
)
sdf = (
    sdf
    .join(
        F.broadcast(
            zone_dim
            .withColumnRenamed("zone_id",   "origin_zone")
            .withColumnRenamed("zone_name", "origin_zone_name")
        ),
        on="origin_zone", how="left",
    )
    .join(
        F.broadcast(
            zone_dim
            .withColumnRenamed("zone_id",   "destination_zone")
            .withColumnRenamed("zone_name", "destination_zone_name")
        ),
        on="destination_zone", how="left",
    )
)

# ── Write (S3A first, local fallback) ─────────────────────────────────────────
output_path = CFG.output_s3
try:
    sdf.write.mode("overwrite").partitionBy("day_type").parquet(output_path)
    print(f"Saved to S3A : {output_path}")
except Exception as _exc:
    print(f"S3A unavailable ({_exc}), writing locally …")
    output_path = CFG.output_local
    os.makedirs(output_path, exist_ok=True)
    sdf.write.mode("overwrite").partitionBy("day_type").parquet(output_path)
    print(f"Saved locally: {output_path}")

print(f"Total rows    : {sdf.count():,}")
print(f"Partitions    : day_type=A (weekday), day_type=B (weekend)")
output_path

## §9 · Validation & Visualisation

### Purpose
Five validation panels confirm that the synthetic dataset is statistically
plausible and geographically consistent with real Casablanca mobility patterns.

### Panel catalogue

| Panel | Chart | Realism check |
|---|---|---|
| **P1** | Histogram — haversine distance | Peak at 2 – 5 km (short intra-city) |
| **P2** | Heatmap — OD matrix (16 × 16) | High-flow corridors match HACA commuter patterns |
| **P3** | Histogram — fare distribution | HACA reference band 25 – 80 MAD mean |
| **P4** | Histogram — duration | Peak 5 – 30 min, exponential tail |
| **P5** | Demand heatmap — hour × day_type | Jumu'ah dip visible on Friday row |
| **P6** | Folium HeatMap — trip origins | Density hotspots at CBD / transport hubs |

### Failure thresholds

| Metric | Threshold | Action on fail |
|---|---|---|
| Distance median | 1.5 – 8.0 km | Yellow warning annotation |
| Fare mean | 25 – 80 MAD | Red warning annotation |
| Duration mean | 300 – 1 800 s | Yellow warning annotation |
| Route coverage | ≥ 85 % | Fail banner + log |

Detailed formal assertions are in §11.

In [ ]:
# ── Panel 1 + 2: Distance distribution & OD flow matrix ──────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Distance histogram
ax = axes[0]
_bins   = np.linspace(0, 40, 80)
_pdist  = porto_distances[(~np.isnan(porto_distances))
                          & (porto_distances > 0.2)
                          & (porto_distances < 80)]
_sdist  = final_pdf["distance_km"].dropna()
_sdist  = _sdist[(_sdist > 0.2) & (_sdist < 80)]
ax.hist(_pdist, bins=_bins, alpha=0.55, density=True,
        label=f"Porto (calibration, n={len(_pdist):,})", color="#4e9af1")
ax.hist(_sdist, bins=_bins, alpha=0.55, density=True,
        label=f"Synthetic Casablanca (n={len(_sdist):,})", color="#f4743b")
ax.set_xlabel("Distance (km)")
ax.set_ylabel("Density")
ax.set_title(f"Trip Distance Distribution  (β={beta_hat:.2f})")
ax.legend()

# OD flow matrix (16×16)
ax = axes[1]
_od_cnt = (final_pdf
           .groupby(["origin_zone", "destination_zone"])
           .size()
           .unstack(fill_value=0))
_im = ax.imshow(_od_cnt.values, cmap="YlOrRd", aspect="auto")
ax.set_xlabel("Destination Zone ID")
ax.set_ylabel("Origin Zone ID")
ax.set_title("OD Flow Matrix (Synthetic Trip Counts)")
plt.colorbar(_im, ax=ax, shrink=0.85)

plt.suptitle("TaaSim Casablanca — Distance & OD Validation", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ── Panel 3 + 4: Hour distribution & Fare distribution ───────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Hour distribution
ax = axes[0]
_hours = pd.to_datetime(final_pdf["timestamp"], unit="s", utc=True).dt.hour
_hcnt  = _hours.value_counts().sort_index()
ax.bar(_hcnt.index, _hcnt.values / _hcnt.values.sum(),
       color="#6c5ce7", alpha=0.85, edgecolor="white", linewidth=0.5)
ax.set_xlabel("Hour of Day (UTC)")
ax.set_ylabel("Share of Trips")
ax.set_title("Casablanca Synthetic Trip Hour Distribution")
ax.set_xticks(range(0, 24, 2))
# Annotate expected peaks
for h, label in [(8, "AM"), (18, "PM")]:
    ax.axvline(h, color="#d63031", linestyle="--", linewidth=1.2, alpha=0.7, label=f"{h}h peak")
ax.legend(fontsize=9)

# Fare distribution
ax = axes[1]
_fare = final_pdf["fare_mad"].dropna()
ax.hist(_fare, bins=60, color="#00b894", alpha=0.85, edgecolor="white", linewidth=0.4)
ax.axvline(_fare.median(), color="#d63031", linestyle="--", linewidth=1.5,
           label=f"Median {_fare.median():.1f} MAD")
ax.set_xlabel("Fare (MAD)")
ax.set_ylabel("Count")
ax.set_title("Petit Taxi Fare Distribution (Casablanca 2024 Tariff)")
ax.legend()

plt.suptitle("TaaSim Casablanca — Temporal & Fare Validation", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# §9 – Extended Validation Panels  P3 · P4 · P5
#   P3  Fare histogram vs HACA reference
#   P4  Duration histogram with expected-range band
#   P5  Demand heatmap (hour × day_of_week) with Jumu'ah annotation
# ─────────────────────────────────────────────────────────────────────────────
import warnings

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
fig.suptitle("§9 — Extended Realism Panels  (P3 · P4 · P5)",
             fontsize=14, fontweight="bold", y=1.01)

# ── P3: Fare distribution ─────────────────────────────────────────────────────
ax3 = axes[0]
if "fare_mad" in final_pdf.columns:
    fare_vals = final_pdf["fare_mad"].dropna()
    ax3.hist(fare_vals, bins=60, color="#1565C0", alpha=0.75, edgecolor="white", lw=0.4)
    ax3.axvspan(25, 80, alpha=0.10, color="#4CAF50",
                label="HACA mean band  [25–80 MAD]")
    ax3.axvline(fare_vals.mean(), color="#D32F2F", lw=1.8, linestyle="--",
                label=f"Synthetic mean = {fare_vals.mean():.1f} MAD")
    ax3.axvline(42, color="#388E3C", lw=1.8, linestyle=":",
                label="HACA 2019 mean = 42 MAD")
    ax3.axvline(38, color="#66BB6A", lw=1.2, linestyle=":",
                label="HACA 2019 median = 38 MAD")
    ok = 25 <= fare_vals.mean() <= 80
    ax3.set_title(f"P3 · Fare Distribution  {'✅' if ok else '⚠️'}", fontsize=12)
    ax3.set_xlabel("Fare (MAD)", fontsize=11)
    ax3.set_ylabel("Trip count", fontsize=11)
    ax3.legend(fontsize=8)
    if not ok:
        warnings.warn(f"P3: mean fare {fare_vals.mean():.1f} MAD outside [25, 80]")
else:
    ax3.text(0.5, 0.5, "fare_mad not available\n(run §7 first)",
             ha="center", va="center", transform=ax3.transAxes, fontsize=11)
    ax3.set_title("P3 · Fare Distribution")

# ── P4: Duration distribution ─────────────────────────────────────────────────
ax4 = axes[1]
if "duration_s" in final_pdf.columns:
    dur_min = final_pdf["duration_s"].dropna() / 60.0
    ax4.hist(dur_min, bins=60, color="#E65100", alpha=0.75, edgecolor="white", lw=0.4)
    ax4.axvspan(5, 30, alpha=0.10, color="#1565C0",
                label="Expected peak band  [5–30 min]")
    ax4.axvline(dur_min.mean(), color="#D32F2F", lw=1.8, linestyle="--",
                label=f"Synthetic mean = {dur_min.mean():.1f} min")
    ok = 5 <= dur_min.mean() <= 30
    ax4.set_title(f"P4 · Trip Duration  {'✅' if ok else '⚠️'}", fontsize=12)
    ax4.set_xlabel("Duration (minutes)", fontsize=11)
    ax4.set_ylabel("Trip count", fontsize=11)
    ax4.set_xlim(left=0)
    ax4.legend(fontsize=8)
    if not ok:
        warnings.warn(f"P4: mean duration {dur_min.mean():.1f} min outside [5, 30]")
else:
    ax4.text(0.5, 0.5, "duration_s not available\n(run §7 first)",
             ha="center", va="center", transform=ax4.transAxes, fontsize=11)
    ax4.set_title("P4 · Trip Duration")

# ── P5: Demand heatmap (hour × day_of_week) ──────────────────────────────────
ax5 = axes[2]
_hour_col = next((c for c in ["trip_hour", "hour"] if c in final_pdf.columns), None)
_dow_col  = next((c for c in ["trip_day_of_week", "day_of_week"] if c in final_pdf.columns), None)

if _hour_col and _dow_col:
    pivot = (
        final_pdf
        .groupby([_dow_col, _hour_col])
        .size()
        .reset_index(name="count")
        .pivot(index=_dow_col, columns=_hour_col, values="count")
        .reindex(columns=range(24), fill_value=0)
        .fillna(0)
    )
    DAY_LABELS = {0:"Mon", 1:"Tue", 2:"Wed", 3:"Thu", 4:"Fri", 5:"Sat", 6:"Sun"}
    im = ax5.imshow(pivot.values, aspect="auto", cmap="YlOrRd",
                    origin="upper", interpolation="nearest")
    ax5.set_yticks(range(len(pivot.index)))
    ax5.set_yticklabels([DAY_LABELS.get(i, str(i)) for i in pivot.index], fontsize=9)
    ax5.set_xticks(range(24))
    ax5.set_xticklabels([str(h) for h in range(24)], fontsize=7, rotation=90)
    ax5.set_xlabel("Hour of day", fontsize=11)
    ax5.set_title("P5 · Demand Heatmap  (hour × day_of_week)", fontsize=12)
    plt.colorbar(im, ax=ax5, label="Trip count", shrink=0.8)

    # Highlight Jumu'ah cell (Friday = index 4, hour = 12)
    if 4 in list(pivot.index) and 12 in list(pivot.columns):
        row_idx = list(pivot.index).index(4)
        col_idx = list(pivot.columns).index(12)
        rect = plt.Rectangle((col_idx - 0.5, row_idx - 0.5), 1, 1,
                              fill=False, edgecolor="#1565C0", lw=2.5)
        ax5.add_patch(rect)
        ax5.text(col_idx, row_idx, "Jumu'ah\n🕌", ha="center", va="center",
                 fontsize=6, color="#1565C0", fontweight="bold")

    # Annotate evening peak box
    for h in [17, 18, 19]:
        for r_idx, dow in enumerate(pivot.index):
            rect = plt.Rectangle((h - 0.5, r_idx - 0.5), 1, 1,
                                  fill=False, edgecolor="#388E3C", lw=1, linestyle="--")
            ax5.add_patch(rect)
else:
    ax5.text(0.5, 0.5, "Temporal columns not available\n(run §6 first)",
             ha="center", va="center", transform=ax5.transAxes, fontsize=11)
    ax5.set_title("P5 · Demand Heatmap")

plt.tight_layout()
_save_path = "validation_extended_panels_p3_p5.png"
plt.savefig(_save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"\nExtended panels P3–P5 saved → {_save_path}")


In [ ]:
# ── Folium interactive heatmap of trip origins ────────────────────────────────
import folium

_center = [zones_gdf["centroid_lat"].mean(), zones_gdf["centroid_lon"].mean()]
m = folium.Map(location=_center, zoom_start=11, tiles="CartoDB positron")

_heat_data = (
    final_pdf[["origin_lat", "origin_lon"]].dropna()
    .sample(n=min(20_000, len(final_pdf)), random_state=42)
    .values.tolist()
)
HeatMap(_heat_data, radius=8, blur=7, min_opacity=0.3).add_to(m)

# Overlay zone polygons
for _, _row in zones_gdf.iterrows():
    folium.GeoJson(
        _row["geometry"].__geo_interface__,
        style_function=lambda x: {
            "fillOpacity": 0.05, "color": "#636e72", "weight": 1.5
        },
        tooltip=_row["zone_name"],
    ).add_to(m)

m

In [ ]:
# ── OSM route overlay — sample of 200 routed trips ───────────────────────────
fig, ax = ox.plot_graph(
    G_casa, node_size=0, edge_linewidth=0.3, edge_color="#dfe6e9",
    figsize=(12, 12), bgcolor="#2d3436", show=False, close=False,
)

_routed_sample = (
    final_pdf[final_pdf["route_generated"]]
    .sample(n=min(200, int(final_pdf["route_generated"].sum())), random_state=42)
)

for _pl_json in _routed_sample["polyline"]:
    try:
        _pl = json.loads(_pl_json) if isinstance(_pl_json, str) else _pl_json
        if not _pl or len(_pl) < 2:
            continue
        ax.plot([p[0] for p in _pl], [p[1] for p in _pl],
                color="#fd79a8", alpha=0.4, linewidth=0.8)
    except Exception:
        pass

ax.set_title("Sample Synthetic OSM Routes — Casablanca", color="white", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── Spatial + quality coverage checks ────────────────────────────────────────
_lon_min_c = zones_gdf["lon_min"].min()
_lon_max_c = zones_gdf["lon_max"].max()
_lat_min_c = zones_gdf["lat_min"].min()
_lat_max_c = zones_gdf["lat_max"].max()

_ok_o = (
    final_pdf["origin_lon"].between(_lon_min_c, _lon_max_c) &
    final_pdf["origin_lat"].between(_lat_min_c, _lat_max_c)
).mean()
_ok_d = (
    final_pdf["destination_lon"].between(_lon_min_c, _lon_max_c) &
    final_pdf["destination_lat"].between(_lat_min_c, _lat_max_c)
).mean()

print("=" * 52)
print("  SPATIAL COVERAGE")
print(f"  Origins inside Casa bbox   : {_ok_o * 100:.2f}%")
print(f"  Destinations inside bbox   : {_ok_d * 100:.2f}%")
print()
print("  ROUTE QUALITY")
print(f"  OSM routes generated       : {final_pdf['route_generated'].sum():,}")
print(f"  Fallback straight-line     : {(~final_pdf['route_generated']).sum():,}")
print(f"  OSM route share            : {final_pdf['route_generated'].mean():.1%}")
print()
print("  DISTANCE STATISTICS (km)")
print(final_pdf["distance_km"].describe().round(3).to_string())
print()
print("  FARE STATISTICS (MAD)")
print(final_pdf["fare_mad"].describe().round(2).to_string())
print()
print("  DURATION STATISTICS (minutes)")
print((final_pdf["duration_sec"] / 60).describe().round(1).to_string())
print("=" * 52)

## §10 · Schema & Statistical Summary

### Purpose
Print the Spark DataFrame schema, per-column descriptive statistics, and the key
simulation KPIs for comparison against real-world Casablanca reference values.

### KPI dashboard

| KPI | Synthetic target | Casablanca reference |
|---|---|---|
| **Total trips** | `config.n_trips` | — |
| **Route coverage** | ≥ 85 % | — |
| **Intra-zone share** | 15 – 30 % | ≈ 20 % (HACA estimate) |
| **Mean distance** | 3.5 – 6.5 km | 4.2 km (HACA 2019) |
| **Median fare** | 25 – 65 MAD | 38 MAD (HACA 2019 median) |
| **Mean fare** | 25 – 80 MAD | 42 MAD (HACA 2019 mean) |
| **Evening / overnight ratio** | ≥ 4.0× | — |
| **β (gravity decay)** | 1.2 – 2.5 | — |

Values outside the target range are printed in ⚠️ bold red.
The formal assertion battery is in §11.

In [ ]:
print("=" * 60)
print("  SPARK DATAFRAME SCHEMA")
print("=" * 60)
sdf.printSchema()

print()
print("=" * 60)
print("  SAMPLE ROWS")
print("=" * 60)
sdf.select(
    "trip_id", "taxi_id", "origin_zone_name", "destination_zone_name",
    "distance_km", "duration_sec", "fare_mad", "day_type", "route_generated",
).show(5, truncate=False)

print()
print("=" * 60)
print("  ZONE-LEVEL TRIP ORIGINS (top 16)")
print("=" * 60)
(sdf.groupBy("origin_zone_name")
    .count()
    .orderBy(F.col("count").desc())
    .show(16, truncate=False))

print()
print("  AGGREGATE STATISTICS")
sdf.select(
    F.count("trip_id").alias("total_trips"),
    F.round(F.avg("distance_km"), 2).alias("avg_distance_km"),
    F.round(F.avg("duration_sec") / 60, 1).alias("avg_duration_min"),
    F.round(F.avg("fare_mad"), 2).alias("avg_fare_mad"),
    F.sum(F.col("route_generated").cast("int")).alias("osm_routed_trips"),
).show(truncate=False)

In [ ]:
def run_pipeline(cfg: SimulationConfig) -> str:
    """Pipeline entry point — for invocation from a DAG or CLI script.

    In practice, execute notebook cells top-to-bottom in Jupyter.
    This function is a documented reference for data pipeline orchestration.
    """
    print("TaaSim Casablanca simulation pipeline complete.")
    print("All sections (§1-§10) must be executed top-to-bottom in Jupyter.")
    try:
        return output_path
    except NameError:
        return cfg.output_local


if __name__ == "__main__":
    final_path = run_pipeline(CFG)
    print(f"Done: {final_path}")

## §11 · Statistical Validation Suite

**What:** Formal assertion battery with binary PASS / FAIL outcomes for  
each realism criterion. Any FAIL prints a `⚠ WARNING` banner (but does  
**not** halt the notebook to allow partial-run inspection).

This section serves as the **quality gate** before promoting synthetic data  
to the TaaSim simulation engine or publishing to the project Parquet store.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# §11 · Statistical Validation Suite
# Formal PASS / FAIL assertion battery aligned to HACA 2019 reference values
# and published Casablanca mobility research.
#
# ⚠️  Any FAIL prints a warning but does NOT halt the notebook — this allows
#     partial-run inspection.  Fix and re-run §11 before promoting data.
# ─────────────────────────────────────────────────────────────────────────────
import warnings
from scipy import stats as sp_stats

_PASS = "✅ PASS"
_FAIL = "⚠️  FAIL"

_results: list[bool] = []

def _check(label: str, condition: bool, detail: str = "") -> bool:
    status  = _PASS if condition else _FAIL
    suffix  = f"   [{detail}]" if detail else ""
    print(f"  {status}  {label}{suffix}")
    if not condition:
        warnings.warn(f"§11 Validation FAIL — {label}  {detail}")
    _results.append(condition)
    return condition

def _section(title: str) -> None:
    print()
    print(f"  {'─'*58}")
    print(f"  {title}")
    print(f"  {'─'*58}")

print("=" * 62)
print("§11  TaaSim-Casablanca · Statistical Validation Suite")
print("     Reference: HACA 2019 Urban Transport Survey, Morocco")
print("=" * 62)

# ── 11.1  Zone coverage ───────────────────────────────────────────────────────
_section("11.1  Zone Coverage  (16 arrondissements required)")
_n_zones = len(zones_gdf)
_orig_zones = final_pdf["origin_zone"].nunique()
_dest_zones = final_pdf["destination_zone"].nunique()
_check("All origin zones present",  _orig_zones == _n_zones, f"{_orig_zones}/{_n_zones}")
_check("All dest zones present",    _dest_zones == _n_zones, f"{_dest_zones}/{_n_zones}")

# ── 11.2  Gravity β plausibility ──────────────────────────────────────────────
_section("11.2  Gravity β Plausibility  (literature: 1.2 – 2.5 for mid-size cities)")
try:
    _check("β ∈ [1.2, 2.5]", 1.2 <= beta_calibrated <= 2.5,
           f"β = {beta_calibrated:.4f}")
except NameError:
    print("  ℹ️   Skipped — beta_calibrated not defined (run §3)")

# ── 11.3  Distance realism ────────────────────────────────────────────────────
_section("11.3  Distance Realism  (HACA 2019: mean 4.2 km, range 0.5 – 18 km)")
_dist_km = (final_pdf["distance_osm"].dropna() / 1_000.0)
_d_med   = _dist_km.median()
_d_p95   = _dist_km.quantile(0.95)
_d_mean  = _dist_km.mean()
_check("Median distance ∈ [1.5, 8.0] km", 1.5 <= _d_med  <= 8.0,  f"median = {_d_med:.2f} km")
_check("Mean   distance ∈ [2.0, 8.0] km", 2.0 <= _d_mean <= 8.0,  f"mean   = {_d_mean:.2f} km")
_check("95th-pct distance < 20 km",        _d_p95 < 20.0,          f"p95    = {_d_p95:.2f} km")

# ── 11.4  Route coverage ──────────────────────────────────────────────────────
_section("11.4  Route Coverage  (target ≥ 85 %)")
_routed   = final_pdf["distance_osm"].notna().sum()
_total    = len(final_pdf)
_coverage = _routed / _total
_check("Route coverage ≥ 85 %", _coverage >= 0.85,
       f"{_coverage*100:.1f} %  ({_routed}/{_total})")

# ── 11.5  Fare realism ────────────────────────────────────────────────────────
_section("11.5  Fare Realism  (Arrêté n° 3-71-19 / HACA 2019: mean 42 MAD)")
if "fare_mad" in final_pdf.columns:
    _fare    = final_pdf["fare_mad"].dropna()
    _f_mean  = _fare.mean()
    _f_med   = _fare.median()
    _f_p5    = _fare.quantile(0.05)
    _f_p95   = _fare.quantile(0.95)
    _check("Mean fare ∈ [25, 80] MAD",           25.0 <= _f_mean <= 80.0, f"mean = {_f_mean:.2f} MAD")
    _check("Median fare ∈ [20, 65] MAD",          20.0 <= _f_med  <= 65.0, f"med  = {_f_med:.2f}  MAD")
    _check("5th-pct fare ≥ 10 MAD (min trip)",   _f_p5  >= 10.0,          f"p5   = {_f_p5:.2f}  MAD")
    _check("95th-pct fare ≤ 200 MAD (legal cap)", _f_p95 <= 200.0,         f"p95  = {_f_p95:.2f} MAD")
else:
    print("  ℹ️   Skipped — fare_mad not in final_pdf (run §7)")

# ── 11.6  Duration realism ────────────────────────────────────────────────────
_section("11.6  Duration Realism  (expected 5 – 30 min petit-taxi)")
if "duration_s" in final_pdf.columns:
    _dur    = final_pdf["duration_sec"].dropna()
    _du_m   = _dur.mean()
    _du_p5  = _dur.quantile(0.05)
    _du_p95 = _dur.quantile(0.95)
    _check("Mean duration ∈ [300, 1800] s  (5–30 min)", 300 <= _du_m  <= 1800, f"mean = {_du_m:.0f} s")
    _check("5th-pct duration ≥ 60 s",                   _du_p5  >= 60,         f"p5   = {_du_p5:.0f} s")
    _check("95th-pct duration ≤ 3600 s  (≤ 60 min)",   _du_p95 <= 3600,        f"p95  = {_du_p95:.0f} s")
else:
    print("  ℹ️   Skipped — duration_s not in final_pdf (run §7)")

# ── 11.7  Temporal realism ────────────────────────────────────────────────────
_section("11.7  Temporal Realism  (HACA: evening peak >> overnight trough)")
_hcol = next((c for c in ["trip_hour","hour"] if c in final_pdf.columns), None)
_dcol = next((c for c in ["trip_day_of_week","day_of_week"] if c in final_pdf.columns), None)
if _hcol:
    _eve = ((final_pdf[_hcol] >= 17) & (final_pdf[_hcol] < 20)).sum()
    _ngt = ((final_pdf[_hcol] >= 1)  & (final_pdf[_hcol] < 5)).sum()
    _ratio = _eve / max(_ngt, 1)
    _check("Evening peak / overnight trough ≥ 4.0×", _ratio >= 4.0, f"ratio = {_ratio:.2f}")
    # Jumu'ah assertion
    if _dcol:
        _fri_noon = ((final_pdf[_dcol] == 4) & (final_pdf[_hcol] == 12)).sum()
        _fri_morn = ((final_pdf[_dcol] == 4) & (final_pdf[_hcol] == 11)).sum()
        if _fri_morn > 0:
            _jq = _fri_noon / _fri_morn
            _check("Jumu'ah dip: Fri 12h < 60 %  of Fri 11h", _jq < 0.60,
                   f"ratio = {_jq:.2f}  ({_fri_noon} vs {_fri_morn} trips)")
        else:
            print("  ℹ️   Jumu'ah check skipped — no Friday 11h trips in sample")
else:
    print("  ℹ️   Skipped — trip_hour not in final_pdf (run §6)")

# ── 11.8  KS-test: synthetic vs Porto distance proxy ──────────────────────────
_section("11.8  KS-Test — Synthetic ↔ Porto Distance Proxy  (p ≥ 0.05 required)")
try:
    _sample  = _dist_km.sample(min(5_000, len(_dist_km)), random_state=42)
    _ks, _kp = sp_stats.ks_2samp(porto_distances_km.values, _sample.values)
    _check("KS p-value ≥ 0.05", _kp >= 0.05,
           f"KS-stat = {_ks:.4f},  p = {_kp:.4f}")
except NameError:
    print("  ℹ️   Skipped — porto_distances_km not defined (run §3)")
except Exception as _exc:
    print(f"  ℹ️   Skipped — {_exc}")

# ── 11.9  Data integrity ──────────────────────────────────────────────────────
_section("11.9  Data Integrity  (no nulls in critical columns)")
for _col in ["origin_lat", "origin_lon", "dest_lat", "dest_lon"]:
    if _col in final_pdf.columns:
        _n = final_pdf[_col].isnull().sum()
        _check(f"No null in {_col}", _n == 0, f"{_n} nulls")
if "polyline" in final_pdf.columns:
    _np = final_pdf["polyline"].isnull().sum()
    _check("No null polylines", _np == 0, f"{_np} nulls")

# ── Summary ───────────────────────────────────────────────────────────────────
print()
print("=" * 62)
_n_pass = sum(_results)
_n_fail = len(_results) - _n_pass
_icon   = "✅" if _n_fail == 0 else "⚠️ "
print(f"  {_icon}  Validation complete:  "
      f"{_n_pass} passed,  {_n_fail} failed  ({len(_results)} total checks)")
if _n_fail > 0:
    print()
    print("  Action required: fix failing checks before promoting data.")
    print("  See §3 (β), §6 (temporal), §7 (fare/speed) for tuning knobs.")
print("=" * 62)


In [ ]:
app_id = spark.sparkContext.applicationId
print(app_id)

In [ ]:
spark.stop()